<a href="https://colab.research.google.com/github/anirbanghoshsbi/.github.io/blob/master/work/err/cross_section_momentum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import duckdb
import pandas as pd

# Path to your DuckDB file in Google Drive
db_path = "/content/drive/MyDrive/All_equities_2020_onwards.duckdb"

# Connect to database
con = duckdb.connect(db_path)

In [3]:
con.execute("SHOW TABLES").fetchall()

[('equities',)]

In [5]:
df = con.execute("SELECT * FROM equities").fetchdf()
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
# Ensure date format
df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'])

# Sort properly
df = df.sort_values(['SYMBOL', 'TIMESTAMP'])

# Keep only equity series
df = df[df['SERIES'] == 'EQ']

In [11]:
df['close_6m_ago'] = df.groupby('SYMBOL')['CLOSE'].shift(126)
df['return_6m'] = (df['CLOSE'] / df['close_6m_ago']) - 1

In [12]:
df['avg_vol_6m'] = (
    df.groupby('SYMBOL')['TOTTRDQTY']
      .rolling(126)
      .mean()
      .reset_index(level=0, drop=True)
)

In [13]:
latest_date = df['TIMESTAMP'].max()

latest_df = df[df['TIMESTAMP'] == latest_date].copy()

In [14]:
latest_df = latest_df[latest_df['avg_vol_6m'] > 1_000_000]

In [15]:
latest_df = latest_df.dropna(subset=['return_6m'])

latest_df = latest_df.sort_values('return_6m', ascending=False)

top_stocks = latest_df[['SYMBOL', 'CLOSE', 'return_6m', 'avg_vol_6m']]

top_stocks.head(20)

,SYMBOL,CLOSE,return_6m,avg_vol_6m
2681046,CUPID,518.10,3.818638,4.542653e+06
2681847,JAYNECOIND,87.82,1.314097,2.882290e+07
2680708,ATHERENERG,754.75,1.271291,3.278972e+06
2681986,SILVER1,220.30,1.125422,1.202230e+06
2680865,SILVER,226.46,1.121206,2.053519e+06
2682884,SBISILVER,221.34,1.106997,2.357938e+06
2683626,SILVERCASE,22.88,1.106814,9.238273e+06
2683274,TATSILV,21.85,1.105010,3.760251e+07
2682419,SILVERBEES,215.43,1.102371,3.720032e+07
2681693,SILVERIETF,225.12,1.102157,7.735541e+06
